# 🛠️ 使用 GitHub 模型的高级工具使用（Python）

## 📋 学习目标

本笔记本展示了使用 Microsoft Agent Framework 与 GitHub 模型集成的高级工具集成模式。您将学习如何创建、管理和编排多个工具，以构建复杂的智能体功能。

**您将掌握：**
- 🔧 **多工具架构**：构建具有多个专用工具的智能体
- 🎯 **工具选择逻辑**：智能体如何为每个任务选择正确的工具
- 📊 **数据处理工具**：创建处理不同数据类型的工具
- 🔗 **工具组合**：组合工具以实现复杂工作流程

## 🎯 关键工具模式

### 工具设计原则
- **单一职责**：每个工具都有明确、专注的目的
- **类型安全**：强类型确保可靠的工具执行
- **错误处理**：优雅的失败和恢复模式
- **可组合性**：工具之间能够良好协作

### 高级工具特性
- **上下文感知**：理解对话上下文的工具
- **数据验证**：输入净化和输出验证
- **性能优化**：高效的工具执行模式
- **可扩展性**：易于添加新的工具功能

## 🔧 技术架构

### 核心组件
- **Microsoft Agent Framework**：具有高级工具支持的 Python 实现
- **GitHub 模型集成**：高性能语言模型访问
- **工具注册系统**：智能体功能的有组织管理
- **错误恢复模式**：工具执行失败的健壮处理

### 工具集成流程
```python
用户请求 → 智能体分析 → 工具选择 → 工具执行 → 响应合成
```

## 🛠️ 演示的工具类别

### 1. **数据生成工具**
- 随机目的地生成器
- 天气信息提供器  
- 旅行成本计算器
- 活动推荐引擎

### 2. **处理工具**
- 文本格式化和验证
- 数据转换工具
- 内容分析功能
- 响应增强工具

### 3. **集成工具**
- 外部 API 连接器
- 文件系统操作
- 数据库查询接口
- 网页抓取工具

## ⚙️ 前提条件和设置

**所需依赖项：**
```bash
pip install agent-framework-core -U
```

**环境配置（.env 文件）：**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**可选 API：**
- 天气服务 API 密钥（用于天气工具）
- 货币转换 API 访问
- 旅行信息服务凭证

## 🎨 设计模式

### 工具工厂模式
- 集中式工具创建和配置
- 一致的工具接口设计
- 易于工具注册和发现

### 命令模式
- 封装的工具执行逻辑
- 复杂操作的撤销/重做功能
- 工具使用的审计日志

### 观察者模式
- 工具执行监控
- 性能指标收集
- 错误报告和警报

## 🚀 最佳实践

- **工具文档**：为智能体理解提供清晰的描述
- **输入验证**：健壮的参数检查和净化
- **输出格式化**：一致、可解析的工具响应
- **错误消息**：有助于调试的有用错误信息
- **性能**：针对响应速度优化的工具执行

准备好构建具有强大工具功能的智能体了吗？让我们创造一些令人惊叹的东西！⚡

In [ ]:
# ! pip install agent-framework-core -U

In [1]:
# 导入 Agent Framework 和工具集成的核心依赖项
# 这为构建具有工具功能的智能体设置了必要的库

import asyncio
import os
import json

from dotenv import load_dotenv  # 用于安全加载环境变量
from random import randint

# 这些是构建支持工具的智能体的核心组件
from agent_framework import ChatAgent           # 主要智能体类
from agent_framework.openai import OpenAIChatClient  # 兼容 OpenAI 的客户端

In [ ]:
load_dotenv()

In [2]:
# 🔑 环境变量验证
# 确保所有必需的 GitHub Models 凭证都已正确配置
print('GITHUB_ENDPOINT:', os.environ.get('GITHUB_ENDPOINT'))
print('GITHUB_TOKEN:', '***' if os.environ.get('GITHUB_TOKEN') else 'Not set')
print('GITHUB_MODEL_ID:', os.environ.get('GITHUB_MODEL_ID'))

GITHUB_ENDPOINT: https://api.siliconflow.cn/v1
GITHUB_TOKEN: ***
GITHUB_MODEL_ID: Qwen/Qwen3-8B


In [3]:
# 🧪 测试 GitHub Models 连接
openai_chat_client = OpenAIChatClient(base_url=os.environ.get('GITHUB_ENDPOINT'), api_key=os.environ.get('GITHUB_TOKEN'), model_id=os.environ.get('GITHUB_MODEL_ID'))

In [4]:
# 🛠️ 定义用于智能体集成的旅行规划工具
# 这些函数提供智能体可以动态调用的特定功能

def get_random_destination() -> str:
    '''
    🎲 随机目的地生成工具
    从精选列表中返回随机选择的旅行目的地
    当客户需要为下一次假期寻找灵感时很有用
    '''
    destinations = [
        'Paris, France',
        'Tokyo, Japan',
        'New York City, USA',
        'London, England',
        'Rome, Italy',
        'Sydney, Australia',
        'Dubai, UAE',
        'Barcelona, Spain',
        'Bangkok, Thailand',
        'Amsterdam, Netherlands',
        'Istanbul, Turkey',
        'Prague, Czech Republic',
        'Santorini, Greece',
        'Reykjavik, Iceland',
        'Marrakech, Morocco',
        'Cape Town, South Africa',
        'Rio de Janeiro, Brazil',
        'Bali, Indonesia'
    ]
    # 🎯 从精选目的地列表中返回随机选择
    return destinations[randint(0, len(destinations) - 1)]

In [5]:
# 🔗 初始化 GitHub Models 聊天客户端以进行智能体通信
# 创建智能体和语言模型之间的主要接口
openai_chat_client = OpenAIChatClient(base_url=os.environ.get('GITHUB_ENDPOINT'), api_key=os.environ.get('GITHUB_TOKEN'), model_id=os.environ.get('GITHUB_MODEL_ID'))

In [6]:
# 🤖 配置旅行智能体身份和行为指令
# 定义智能体的个性、能力和操作指南

AGENT_NAME = 'TravelAgent'

AGENT_INSTRUCTIONS = '''You are a helpful AI Agent that can help plan vacations for customers at random destinations
'''

In [7]:
agent = ChatAgent(
        name = AGENT_NAME,
        chat_client=openai_chat_client,
        instructions=AGENT_INSTRUCTIONS,
        tools=[get_random_destination]
)

In [8]:
thread = agent.get_new_thread()

In [9]:
response1 = await agent.run('Plan me a day trip', thread=thread)

In [10]:
last_message = response1.messages[-1]
text_content = last_message.contents[0].text
print('Travel plan:')
print(text_content)

Travel plan:
Here's a suggested day trip plan for Dubai, UAE:

**Morning:**
- Start your day with a visit to the iconic Burj Khalifa, the world's tallest building. Take the elevator to the observation deck on the 124th floor for breathtaking views of the city.
- Explore the nearby Dubai Mall, which is one of the world's largest malls. It features a wide range of shopping, dining, and entertainment options, including an aquarium and ice rink.

**Afternoon:**
- Head to the desert for a Dubai Desert Safari. This activity includes a dune bashing ride, a camel ride, and a traditional Bedouin-style picnic with shisha, dinner, and entertainment like belly dancing.

**Evening:**
- Return to the city and head to the Dubai Fountain, located at the heart of the Dubai Mall. Enjoy the stunning water display and fountain show, especially during the evening when it's illuminated with colorful lights.
- Finish your day with a walk along the Dubai Creek, where you can visit the traditional wooden dhows